# Matplotlib

A list of 1024 measurement outcomes is data. A histogram of those outcomes is understanding. Matplotlib is the standard visualization library for scientific Python, and it produces the figures that appear in quantum computing papers, textbooks, and research presentations.

This notebook accompanies **Lesson 8** of *Python Programming for Quantum Technology I*. Topics covered:
- The Figure / Axes model
- Line plots: Rabi oscillations and probability evolution
- Histograms: shot noise and measurement distributions
- Bar charts: quantum state visualization with phase coloring

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Consistent plot style throughout the notebook
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

---
## 1. The Matplotlib Model

Every plot lives in a **Figure** containing one or more **Axes**. Use the `ax.method()` style throughout: it is explicit about which axes you are drawing on.

In [ ]:
# One figure, one axes
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot([0, 1, 2, 3], [0, 1, 4, 9], color="steelblue", linewidth=2)
ax.set_xlabel("x")
ax.set_ylabel("x²")
ax.set_title("A simple line plot")
plt.tight_layout()
plt.show()

In [ ]:
# Multiple axes: 1 row, 2 columns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))

x = np.linspace(0, 2 * np.pi, 300)

ax1.plot(x, np.sin(x), color="steelblue", linewidth=2)
ax1.set_title("sin(x)")
ax1.set_xlabel("x")

ax2.plot(x, np.cos(x), color="tomato", linewidth=2)
ax2.set_title("cos(x)")
ax2.set_xlabel("x")

plt.tight_layout()
plt.show()

---
## 2. Line Plots

`ax.plot(x, y)` is the core function. Use it whenever the x-axis is continuous or ordered and the values change smoothly.

In [ ]:
x = np.linspace(0, 2 * np.pi, 300)

fig, ax = plt.subplots(figsize=(8, 3.5))

ax.plot(x, np.sin(x),       color="steelblue", linewidth=2,   linestyle="-",  label="sin(x)")
ax.plot(x, np.cos(x),       color="tomato",    linewidth=2,   linestyle="--", label="cos(x)")
ax.plot(x, np.sin(x)**2,    color="seagreen",  linewidth=1.5, linestyle=":",  label="sin²(x)")

ax.axhline(0, color="gray", linewidth=0.8, linestyle="-")

ax.set_xlabel("x", fontsize=12)
ax.set_ylabel("f(x)", fontsize=12)
ax.set_title("Line styles and colors", fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

### Quantum Example: Rabi Oscillations

When a qubit is driven by a resonant pulse, its state oscillates between $|0\rangle$ and $|1\rangle$ at the Rabi frequency $\Omega$:

$$P_{|1\rangle}(t) = \sin^2\!\left(\frac{\Omega t}{2}\right)$$

In [ ]:
Omega = 2 * np.pi   # one full oscillation per unit time
t = np.linspace(0, 4, 500)

prob_one  = np.sin(Omega * t / 2)**2
prob_zero = np.cos(Omega * t / 2)**2

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(t, prob_one,  color="steelblue", linewidth=2.5, label=r"$P(|1\rangle)$")
ax.plot(t, prob_zero, color="tomato",    linewidth=2.5, label=r"$P(|0\rangle)$", linestyle="--")
ax.axhline(0.5, color="gray", linewidth=1, linestyle=":", alpha=0.7)

# Mark the pi-pulse time (full inversion)
t_pi = 1.0
ax.axvline(t_pi, color="goldenrod", linewidth=1.5, linestyle="--", alpha=0.8, label=r"$\pi$-pulse")

ax.set_xlabel(r"Time ($1/\Omega$)", fontsize=12)
ax.set_ylabel("Probability", fontsize=12)
ax.set_title("Rabi Oscillations", fontsize=14)
ax.set_ylim(-0.05, 1.1)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

### Exercise 2.1

**Damped Rabi oscillations.** In a real system, decoherence causes the oscillations to decay over time. A simple model multiplies the ideal Rabi oscillation by an exponential decay:

$$P_{|1\rangle}(t) = \sin^2\!\left(\frac{\Omega t}{2}\right) \cdot e^{-t/T_1}$$

Plot the damped and ideal Rabi curves on the same axes for:
- $\Omega = 2\pi$, $t \in [0, 5]$
- $T_1 = 1.5$ (fast decay) and $T_1 = 5.0$ (slow decay)

Use three different colors and a legend. Add a horizontal dashed line at $P = 0.5$. Label the axes and add a title.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Your code here


---
## 3. Histograms: Visualizing Quantum Measurement

`ax.hist(data, bins=n)` divides the data range into bins and counts how many values fall in each. Use `density=True` to normalize so the histogram integrates to 1.

In [ ]:
rng = np.random.default_rng(42)
data = rng.normal(loc=0.7, scale=0.05, size=800)

fig, ax = plt.subplots(figsize=(6, 4))

ax.hist(data, bins=30, color="steelblue", edgecolor="white", alpha=0.85)

ax.set_xlabel("Value", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Basic histogram", fontsize=13)

plt.tight_layout()
plt.show()

### Quantum Example: Shot Noise

If a qubit has true probability $p = 0.7$ of collapsing to $|1\rangle$, the fraction of $|1\rangle$ outcomes across $n_\text{shots}$ repetitions is random. For large $n_\text{shots}$, this fraction follows a normal distribution with mean $p$ and standard deviation $\sqrt{p(1-p)/n_\text{shots}}$.

In [ ]:
from scipy.stats import norm

rng = np.random.default_rng(42)
p_true   = 0.7
n_shots  = 200
n_expts  = 2000

fractions = rng.binomial(n_shots, p_true, size=n_expts) / n_shots

fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(fractions, bins=40, density=True,
        color="steelblue", edgecolor="white", alpha=0.85,
        label=f"{n_expts} experiments")

# Theoretical normal distribution
sigma = np.sqrt(p_true * (1 - p_true) / n_shots)
x = np.linspace(fractions.min() - 0.02, fractions.max() + 0.02, 300)
ax.plot(x, norm.pdf(x, p_true, sigma),
        color="tomato", linewidth=2.5, label="Theoretical normal")

ax.axvline(p_true, color="gray", linewidth=1.5, linestyle="--",
           label=f"True $p = {p_true}$")

ax.set_xlabel(f"Fraction of $|1\\rangle$ outcomes ({n_shots} shots)", fontsize=11)
ax.set_ylabel("Probability density", fontsize=11)
ax.set_title("Shot Noise Distribution", fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

### Comparing Shot Counts: Overlapping Histograms

Plot two histograms on the same axes with `alpha < 1` so the overlap is visible.

In [ ]:
rng = np.random.default_rng(7)
p_true = 0.7
n_expts = 3000

n_low  = 20
n_high = 500

frac_low  = rng.binomial(n_low,  p_true, size=n_expts) / n_low
frac_high = rng.binomial(n_high, p_true, size=n_expts) / n_high

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(frac_low,  bins=40, density=True, alpha=0.6,
        color="steelblue", edgecolor="white", label=f"$n = {n_low}$ shots")
ax.hist(frac_high, bins=40, density=True, alpha=0.6,
        color="tomato",    edgecolor="white", label=f"$n = {n_high}$ shots")

ax.axvline(p_true, color="gray", linewidth=1.5, linestyle="--", label=f"True $p = {p_true}$")

ax.set_xlabel(r"Fraction of $|1\rangle$ outcomes", fontsize=11)
ax.set_ylabel("Probability density", fontsize=11)
ax.set_title("Effect of Shot Count on Measurement Precision", fontsize=12)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

### Exercise 3.1

Simulate 5000 measurements of a qubit with true probability $p = 0.3$ of measuring $|1\rangle$, with $n_\text{shots} = 100$ per experiment.

1. Generate the fraction-of-$|1\rangle$ data using `rng.binomial`.
2. Plot a normalized histogram with 35 bins.
3. Overlay the theoretical normal distribution.
4. Add a vertical line at the true value $p = 0.3$.
5. Compute and print the empirical mean and standard deviation of the fractions. Compare the standard deviation to the theoretical value $\sqrt{p(1-p)/n_\text{shots}}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Your code here


---
## 4. Bar Charts: Visualizing a Quantum State

`ax.bar(labels, heights)` draws a vertical bar for each label-height pair. It is the standard way to display the probability distribution of a quantum state.

In [ ]:
labels = ["|00⟩", "|01⟩", "|10⟩", "|11⟩"]
probs  = [0.25, 0.25, 0.25, 0.25]

fig, ax = plt.subplots(figsize=(5, 3.5))

ax.bar(labels, probs, color="steelblue", edgecolor="white", width=0.6)

ax.set_xlabel("Basis state", fontsize=12)
ax.set_ylabel("Probability", fontsize=12)
ax.set_title("Equal superposition: 2 qubits", fontsize=13)
ax.set_ylim(0, 0.4)

plt.tight_layout()
plt.show()

### Quantum Example: 3-Qubit State with Annotations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

s = 1 / np.sqrt(2)

# H on q0 only: (|0> + |1>)/sqrt(2) x |0> x |0> = (|000> + |100>)/sqrt(2)
amplitudes = np.zeros(8, dtype=complex)
amplitudes[0b000] = s
amplitudes[0b100] = s

n_qubits = 3
labels = [f"|{format(k, f'0{n_qubits}b')}⟩" for k in range(8)]
probs  = np.abs(amplitudes)**2

fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.bar(labels, probs, color="steelblue", edgecolor="white", width=0.6)

# Annotate non-zero bars
for bar, prob in zip(bars, probs):
    if prob > 0.01:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.012,
                f"{prob:.2f}",
                ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_xlabel("Basis state", fontsize=12)
ax.set_ylabel("Probability", fontsize=12)
ax.set_title(r"$(H \otimes I \otimes I)\,|000\rangle$", fontsize=14)
ax.set_ylim(0, 0.7)

plt.tight_layout()
plt.show()

### Phase Coloring

Probability alone does not capture the full quantum state: two states can have identical probabilities but different phases. The convention is to color bars by the phase angle of the amplitude using a cyclic colormap.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Equal probability, four different phases
phases_deg = [0, 90, 180, 270]
amplitudes = np.array([
    0.5 * np.exp(1j * np.radians(phi)) for phi in phases_deg
], dtype=complex)

probs  = np.abs(amplitudes)**2
phases = np.angle(amplitudes)          # in radians, range [-pi, pi]

cmap   = cm.hsv
colors = [cmap((phi + np.pi) / (2 * np.pi)) for phi in phases]
labels = ["|00⟩", "|01⟩", "|10⟩", "|11⟩"]

fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(labels, probs, color=colors, edgecolor="white", width=0.6)

for bar, phi_deg in zip(bars, phases_deg):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.008,
            f"{phi_deg}°",
            ha="center", va="bottom", fontsize=10)

ax.set_xlabel("Basis state", fontsize=12)
ax.set_ylabel("Probability", fontsize=12)
ax.set_title("Equal probability, varying phase\n(color = phase angle)", fontsize=12)
ax.set_ylim(0, 0.38)

plt.tight_layout()
plt.show()

### Grouped Bar Charts: Before and After a Circuit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n_qubits = 3
labels = [f"|{format(k, f'0{n_qubits}b')}⟩" for k in range(2**n_qubits)]

probs_before = np.array([1, 0, 0, 0, 0, 0, 0, 0], dtype=float)  # |000>
probs_after  = np.ones(8) / 8                                     # equal superposition

x     = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))

ax.bar(x - width/2, probs_before, width,
       color="steelblue", edgecolor="white", alpha=0.9, label=r"Before: $|000\rangle$")
ax.bar(x + width/2, probs_after,  width,
       color="tomato",    edgecolor="white", alpha=0.9, label=r"After: $H^{\otimes 3}|000\rangle$")

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=10)
ax.set_ylabel("Probability", fontsize=12)
ax.set_title(r"Effect of $H \otimes H \otimes H$ on $|000\rangle$", fontsize=13)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

### Exercise 4.1

**Visualizing a Bell state circuit.**

The Bell state $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ is prepared by:
1. Applying H to qubit 0: $|00\rangle \to \frac{1}{\sqrt{2}}(|00\rangle + |10\rangle)$
2. Applying CNOT (control=q0, target=q1): flips q1 when q0=1

Using NumPy arrays and the gate matrices from Lesson 7:
1. Compute the state at each step: initial, after H, after CNOT.
2. Plot a grouped bar chart showing the probability distribution at all three stages side by side.
3. Label each group with its basis state and each bar cluster with the circuit step.

The final state should show only $|00\rangle$ and $|11\rangle$ with probability 0.5 each.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

s = 1 / np.sqrt(2)

# Gate matrices
H    = np.array([[s, s], [s, -s]], dtype=complex)
I    = np.eye(2, dtype=complex)
CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)

# Your code here


---
## Summary

| Plot type | Function | Key arguments | When to use |
|-----------|----------|---------------|-------------|
| Line plot | `ax.plot(x, y)` | `color`, `linewidth`, `linestyle`, `label`, `alpha` | Continuous or ordered data |
| Histogram | `ax.hist(data, bins=n)` | `density`, `color`, `edgecolor`, `alpha`, `label` | Distribution of repeated values |
| Bar chart | `ax.bar(labels, heights)` | `color`, `edgecolor`, `width`, `alpha` | Discrete probability distributions |

**Common operations across all plot types:**

```python
ax.set_xlabel("label")         # x-axis label
ax.set_ylabel("label")         # y-axis label
ax.set_title("title")          # plot title
ax.set_ylim(low, high)         # y-axis range
ax.legend()                    # show legend
ax.axhline(y)                  # horizontal reference line
ax.axvline(x)                  # vertical reference line
ax.text(x, y, "string")        # annotate a point
plt.tight_layout()             # prevent clipping
plt.show()                     # display the figure
fig.savefig("name.png",        # save to file
            dpi=150,
            bbox_inches="tight")
```

**What comes next:** This lesson completes *Python Programming for Quantum Technology I*. You now have the full foundation: Python basics, control flow, functions, data structures, modules, OOP, NumPy, and Matplotlib. The next course applies these tools directly to quantum circuits, state simulation, noise modeling, and real quantum algorithms.

---
## Challenge Problem

**A complete quantum state visualization dashboard.**

Build a function `plot_state(amplitudes, n_qubits, title="")` that produces a 1x2 figure:
- **Left panel:** a bar chart of probabilities ($|\alpha_k|^2$), with bars colored by phase angle using the `hsv` colormap, and each bar annotated with its phase in degrees.
- **Right panel:** a scatter plot of the amplitudes in the complex plane (real part on x, imaginary part on y), with each point labeled by its basis state and sized proportional to its probability.

Test it on:
1. The equal superposition state $H^{\otimes 2}|00\rangle$ (uniform probability, all phases zero).
2. The state with amplitudes $[0.5,\, 0.5i,\, -0.5,\, -0.5i]$ (uniform probability, phases $0°, 90°, 180°, 270°$).
3. The Bell state $|\Phi^+\rangle$ (only two nonzero amplitudes).

For the scatter plot, add the unit circle as a light gray reference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

def plot_state(amplitudes, n_qubits, title=""):
    """
    Visualize a quantum state as a probability bar chart (phase-colored)
    and a complex plane scatter plot.
    """
    # Your code here
    pass


# Test states
s = 1 / np.sqrt(2)

equal_superposition = np.array([0.5, 0.5, 0.5, 0.5], dtype=complex)
phase_state = np.array([0.5, 0.5j, -0.5, -0.5j], dtype=complex)
bell_plus   = np.array([s, 0, 0, s], dtype=complex)

plot_state(equal_superposition, n_qubits=2, title="Equal superposition")
plot_state(phase_state,         n_qubits=2, title="Varying phase")
plot_state(bell_plus,           n_qubits=2, title=r"$|\Phi^+\rangle$")